In [1]:
import sys
sys.path.extend(["/home/user/wangtao/TRUECAM/TITAN"])

import numpy as np
import pandas as pd
import torch
import yaml
import pickle
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
from huggingface_hub import hf_hub_download
from titan.eval_linear_probe import train_and_evaluate_logistic_regression_with_val


prefix_path = Path("/home/user/wangtao/TRUECAM/TITAN")

with open(prefix_path / "datasets/config_tcga-ot.yaml") as f:
    task_config = yaml.load(f, Loader=yaml.FullLoader)
target = task_config["target"]
label_dict = task_config["label_dict"]

slide_feature_path = hf_hub_download("MahmoodLab/TITAN", filename="TCGA_TITAN_features.pkl")
with open(slide_feature_path, "rb") as f:
    data = pickle.load(f)
embeddings_df = pd.DataFrame({"slide_id": data["filenames"],
                               "embeddings": list(data["embeddings"])})

def load_split(split_name):
    split = pd.read_csv(prefix_path / f"datasets/tcga-ot_{split_name}.csv")
    return pd.merge(embeddings_df, split, on="slide_id")

train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")


train_data = np.stack(train_df.embeddings.values)
train_labels = train_df[target].map(label_dict).values
val_data = np.stack(val_df.embeddings.values)
val_labels = val_df[target].map(label_dict).values
test_data = np.stack(test_df.embeddings.values)
test_labels = test_df[target].map(label_dict).values

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print(f"Total classes: {len(label_dict)}")

/home/user/.conda/envs/gigapath/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Train: 8226  |  Val: 1612  |  Test: 1348
Total classes: 46


**Here we use half classes of TCGA-OT as ID and the other half as OOD for illustration purpose**

In [2]:
all_classes = np.unique(train_labels)
num_id = len(all_classes) // 2  # first half → ID, second half → OOD

id_classes = all_classes[:num_id]
ood_classes = all_classes[num_id:]
print(f"ID classes: {len(id_classes)}  |  OOD classes: {len(ood_classes)}")


id_mask_train = np.isin(train_labels, id_classes)
id_mask_val   = np.isin(val_labels, id_classes)

X_train = train_data[id_mask_train]
y_train = train_labels[id_mask_train]
X_val   = val_data[id_mask_val]
y_val   = val_labels[id_mask_val]

class_remap = {orig: new for new, orig in enumerate(id_classes)}
y_train = np.array([class_remap[y] for y in y_train])
y_val   = np.array([class_remap[y] for y in y_val])

print(f"ID train: {len(X_train)}  |  ID val: {len(X_val)}")

id_mask_test  = np.isin(test_labels, id_classes)
X_test_id_only = test_data[id_mask_test]
y_test_id_only = test_labels[id_mask_test]
y_test_id_only = np.array([class_remap[y] for y in y_test_id_only])

_, outputs = train_and_evaluate_logistic_regression_with_val(
    X_train, y_train, X_val, y_val,
    X_val, y_val, log_spaced_values=None
)
clf = outputs["lr_model"]
print(f"ID val accuracy: {clf.score(X_val, y_val):.4f}")

ood_mask_test = np.isin(test_labels, ood_classes)

X_test_id  = test_data[id_mask_test]
X_test_ood = test_data[ood_mask_test]

X_test = np.concatenate([X_test_id, X_test_ood], axis=0)
y_ood  = np.concatenate([np.zeros(len(X_test_id)),      # 0 = ID
                          np.ones(len(X_test_ood))])     # 1 = OOD

print(f"Test ID: {len(X_test_id)}  |  Test OOD: {len(X_test_ood)}")


probs = clf.predict_proba(X_test)
max_prob = probs.max(axis=1)        
ood_score = -max_prob               

auroc = roc_auc_score(y_ood, ood_score)
aupr  = average_precision_score(y_ood, ood_score)

print(f"\n--- OOD Detection (Max Probability) ---")
print(f"AUROC:         {auroc:.4f}")
print(f"AUPR:          {aupr:.4f}")


ID classes: 23  |  OOD classes: 23
ID train: 5208  |  ID val: 829


Finding best C: 100%|██████████| 45/45 [01:35<00:00,  2.12s/it]

Best C: 17.78279410038923
ID val accuracy: 0.8770
Test ID: 730  |  Test OOD: 618

--- OOD Detection (Max Probability) ---
AUROC:         0.8648
AUPR:          0.8408


**Feature based UQ via SNGP on the feature space**

In [3]:
import torch.nn as nn
from sngp_wrapper.covert_utils import replace_layer_with_gaussian


class TitanGaussianProcess(nn.Module):
    
    def __init__(self):
        super(TitanGaussianProcess, self).__init__()
        self.classifier = nn.Linear(768, 1)
        
    def forward(self, x, **kwargs):
        return self.classifier(x, **kwargs)
    
GP_KWARGS = {
    'num_inducing': 768, 'gp_scale': 1.0, 'gp_bias': 0.,
    'gp_kernel_type': 'rbf', 'gp_input_normalization': False,
    'gp_cov_discount_factor': -1, 'gp_cov_ridge_penalty': 1.,
    'gp_output_bias_trainable': True, 'gp_scale_random_features': False,
    'gp_use_custom_random_features': True, 'gp_random_feature_type': 'orf',
    'gp_output_imagenet_initializer': False,
}

n_features, n_classes = X_train.shape[1], len(id_classes)

gp_model = TitanGaussianProcess()
replace_layer_with_gaussian(container=gp_model, signature="classifier", **GP_KWARGS)
gp_model.classifier.reset_covariance_matrix()

with torch.no_grad():
    _ = gp_model(torch.from_numpy(X_train).float(),
                 return_random_features=False, return_covariance=False,
                 update_precision_matrix=True, update_covariance_matrix=False)
gp_model.classifier.update_covariance_matrix()

with torch.no_grad():
    out, cov = gp_model(torch.from_numpy(X_test).float(),
                        return_random_features=False, return_covariance=True,
                        update_precision_matrix=False, update_covariance_matrix=False)
gp_uncertainty = torch.diagonal(cov).numpy() 

gp_auroc = roc_auc_score(y_ood, gp_uncertainty)
gp_aupr  = average_precision_score(y_ood, gp_uncertainty)


print(f"\n--- OOD Detection (GP Uncertainty) ---")
print(f"AUROC:         {gp_auroc:.4f}")
print(f"AUPR:          {gp_aupr:.4f}")



--- OOD Detection (GP Uncertainty) ---
AUROC:         0.8872
AUPR:          0.8852
